# Первое обучение нейросети

CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

В прошлом параграфе мы получили ответ сети вручную: умножили входы на веса, сложили, применили активацию. Но сами веса мы просто назвали — взяли готовые числа, при которых сеть решает XOR. Откуда они берутся на самом деле?

Их находит обучение. Сеть начинает со случайных весов, смотрит на примеры с известными ответами и понемногу себя исправляет. Разберём это на задаче распознавания рукописных цифр: узнать по картинке, какая цифра на ней написана.

После занятия вы сможете:

- Обучить небольшую сеть на размеченных картинках и измерить её точность.
- Объяснить, что происходит внутри `fit`: откуда берётся ошибка и как меняются веса.
- Отличить результат на обучающих примерах от честной оценки на новых.

План: посмотрим на данные и превратим картинку в 64 числа. Обучим сеть отличать 3 от 8 и прочитаем её ответы. Разберём, что делает `fit`: ошибка, производная, шаг. Затем перейдём сразу к десяти цифрам и честно сравним сеть с моделями из первой главы.

Понадобится немного: умение считать ответ сети по весам из [прошлого параграфа](../01-from-linear-model-to-neural-network/content.md) и привычка проверять модель на примерах, которых она не видела. Обучать будем `MLPClassifier` из scikit-learn — тем же вызовом `fit`, что и модели первой главы.

## Задача: 3 или 8

Возьмём набор рукописных цифр, который входит в scikit-learn: 1797 картинок размером 8 на 8 точек. Каждая точка — число от 0 до 16: 0 — пустое место, 16 — самый тёмный штрих.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print(digits.images.shape)  # (картинок, строк, столбцов)
print(digits.target[3])     # какая цифра на картинке с номером 3
print(digits.images[3])     # сама картинка: 8 строк по 8 чисел


Вывод:

```text
(1797, 8, 8)
3
[[ 0.  0.  7. 15. 13.  1.  0.  0.]
 [ 0.  8. 13.  6. 15.  4.  0.  0.]
 [ 0.  2.  1. 13. 13.  0.  0.  0.]
 [ 0.  0.  2. 15. 11.  1.  0.  0.]
 [ 0.  0.  0.  1. 12. 12.  1.  0.]
 [ 0.  0.  0.  0.  1. 10.  8.  0.]
 [ 0.  0.  8.  4.  5. 14.  9.  0.]
 [ 0.  0.  7. 13. 13.  9.  0.  0.]]
```

Присмотритесь к таблице: края заполнены нулями — цифра не касается рамки, — а крупные числа идут сверху вниз и в двух средних строках сдвигаются вправо. Это и есть тройка, записанная числами. Человек видит форму, а сеть получит те же 64 числа: больше ей знать неоткуда.

Сеть принимает не таблицу, а строку чисел. Поэтому 8 строк по 8 чисел выписывают подряд в один вектор длины 64: сначала первая строка картинки, затем вторая и так далее. Такую операцию называют **вытягиванием** картинки в вектор; в `digits.data` цифры уже лежат в таком виде. Порядок чисел при этом всегда один и тот же, поэтому каждый вход сети отвечает за свою точку картинки.



*Рисунок 1 — От картинки к ответу сети*

Начнём с двух классов: будем отличать тройку от восьмёрки. Оставим только эти цифры и разделим все числа на 16, чтобы признаки лежали в отрезке от 0 до 1.

In [ ]:
import numpy as np

mask = (digits.target == 3) | (digits.target == 8)
X = digits.data[mask] / 16                   # 64 числа от 0 до 1 на картинку
y = (digits.target[mask] == 8).astype(int)   # 1 — восьмёрка, 0 — тройка
print(X.shape, y.shape)
print(np.bincount(y))                        # сколько троек и восьмёрок


Вывод:

```text
(357, 64) (357,)
[183 174]
```

Зачем это деление? Выравнивать признаки между собой здесь не нужно: все 64 — яркости точек, они с самого начала измерены одинаково. Дело в другом. Сеть умножает входы на веса и складывает результаты, и чем крупнее входные числа, тем крупнее суммы внутри сети. Со случайными весами она начинает с очень уверенных и при этом неверных ответов, а значит, с большой ошибки: на наших данных 0,74 с делением против 2,46 без него.

На итог в этой задаче деление не влияет: обе сети дают на проверке одинаковые 0,981. Польза видна при более крупном шаге обучения. Если поставить `learning_rate_init=0.1`, сеть с делением даёт 0,991 при любых начальных весах, а без деления результат скачет: на восьми разных `random_state` он равен то 0,99, то примерно 0,5 — это уровень случайного угадывания. Такое приведение входов к небольшим числам называют **масштабированием**.

Классы почти равны по размеру: 183 тройки и 174 восьмёрки. Отложим часть примеров для проверки — модель не увидит их при обучении.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)
print(np.bincount(y_test))


Вывод:

```text
(249, 64) (108, 64)
[55 53]
```

На обучение ушло 249 картинок, на проверку — 108. Параметр `stratify=y` сохранил соотношение классов в обеих частях: 55 троек и 53 восьмёрки.

Прежде чем обучать сеть, узнаем, что даёт самый простой ответ: всегда называть один и тот же класс. Это **отправная точка**, с которой сравнивают любую модель.

In [ ]:
from sklearn.dummy import DummyClassifier

constant = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(round(constant.score(X_test, y_test), 3))


Вывод:

```text
0.509
```

Постоянный ответ верен примерно в половине случаев, потому что классы почти равны по размеру. Значит, всё, что модель получит выше 0,509, она заработала сама.

## Обучаем сеть тремя строками

`MLPClassifier` — это та самая сеть из прошлого параграфа: входы, скрытый слой и выход. Создадим её и обучим.

In [ ]:
from sklearn.neural_network import MLPClassifier

net = MLPClassifier(
    hidden_layer_sizes=(16,),  # один скрытый слой из 16 нейронов
    max_iter=200,              # 200 проходов по обучающим картинкам
    random_state=0,            # чтобы результат повторялся
)
net.fit(X_train, y_train)
print(round(net.score(X_train, y_train), 3))
print(round(net.score(X_test, y_test), 3))


Вывод:

```text
0.996
0.981
```

Создание и обучение заняли три строки, причём само обучение — один вызов `fit`. На обучающих картинках сеть верна в 99,6 % случаев, на отложенных — в 98,1 %, то есть ошиблась в 2 картинках из 108. Число входов и выходов `MLPClassifier` определил сам: 64 признака на входе, два класса — один выход с вероятностью. Мы задали только скрытый слой.



> **Примечание**
> 
> Вместе с ответом scikit-learn напечатает `ConvergenceWarning`: 200 эпох закончились раньше, чем ошибка перестала уменьшаться. Это не ошибка в коде — сеть просто не доучилась до конца. В задании 1 видно, что при `max_iter=500` обучение останавливается само.


Посмотрим, сколько в такой сети весов.

In [ ]:
print([c.shape for c in net.coefs_])       # таблицы весов между слоями
print([b.shape for b in net.intercepts_])  # смещения нейронов


Вывод:

```text
[(64, 16), (16, 1)]
[(16,), (1,)]
```

Между входом и скрытым слоем лежит таблица $64\times16$, между скрытым слоем и выходом — $16\times1$; к ним добавляются 16 и 1 смещение. Всего $64\cdot16+16+16\cdot1+1=1057$ чисел. Все они были случайными до `fit` и подобраны после него.

Как и логистическая регрессия, сеть отвечает вероятностью. Метод `predict_proba` возвращает по столбцу на класс; второй столбец — вероятность восьмёрки.

In [ ]:
proba = net.predict_proba(X_test)[:, 1]  # вероятность восьмёрки
for i in range(5):
    print(y_test[i], round(float(proba[i]), 3))


Вывод:

```text
0 0.003
0 0.388
0 0.009
1 0.997
1 0.989
```

Слева — настоящий ответ, справа — вероятность восьмёрки. В четырёх случаях сеть уверена: вероятности 0,003, 0,009, 0,997 и 0,989. Вторая картинка — тройка, но вероятность восьмёрки у неё 0,388. Класс определяется по порогу 0,5, поэтому ответ верный, но уверенности мало: эта тройка чем-то похожа на восьмёрку.

Посмотрим на две картинки, где сеть всё-таки ошиблась.

In [ ]:
wrong = np.where(net.predict(X_test) != y_test)[0]
print(len(wrong))
for i in wrong:
    print(y_test[i], round(float(proba[i]), 3))


Вывод:

```text
2
0 0.817
0 0.512
```

Обе ошибки одного вида: тройку сеть приняла за восьмёрку. В первом случае она была уверена (0,817), во втором едва перешла порог (0,512). Посмотрим на сами картинки.



*Рисунок 2 — Две картинки, на которых сеть ошиблась*

Обе тройки написаны так, что верх получился широким и закруглённым: на сетке 8 на 8 этого хватает, чтобы цифра стала похожа на восьмёрку. Остальные 106 картинок из 108 сеть разобрала верно.

## Что делает `fit`

Снаружи `fit` — одна строка, внутри — цикл из трёх действий, который повторяется много раз:

1. **Прогноз.** Сеть считает ответ для обучающих картинок — умножает входы на веса, складывает, применяет активацию.
2. **Ошибка.** Ответы сравниваются с правильными, и получается одно число: насколько сеть неправа.
3. **Поправка.** Каждый вес немного меняется в ту сторону, где ошибка меньше.



*Рисунок 3 — Цикл обучения внутри `fit`*

Так же подбирают коэффициенты и простые модели из первой главы. Для сети не изменилось ничего, кроме числа весов: вместо двух-трёх их 1057.

### Ошибка

Ошибку меряют по вероятности, которую сеть дала правильному ответу. Если на картинке восьмёрка, а сеть дала ей вероятность $p$, штраф равен $-\ln p$: чем меньше вероятность у правильного ответа, тем больше штраф. При $p=0{,}9$ штраф $-\ln0{,}9\approx0{,}105$, при $p=0{,}5$ — около $0{,}693$, при $p=0{,}1$ — уже $2{,}303$. Уверенная ошибка стоит дорого, а уверенный верный ответ почти ничего. Средний штраф по всем обучающим картинкам обозначим $L$: это и есть та ошибка, которую обучение уменьшает.

### Поправка весов

Направление поправки подсказывает производная. Обозначим через $L'(w)$ производную ошибки по весу $w$: она показывает, как изменится ошибка, если чуть увеличить этот вес. Если производная положительна, вес надо уменьшить; если отрицательна — увеличить. Поэтому от веса отнимают производную, умноженную на небольшое число $\eta$ — **скорость обучения**:

$$w \leftarrow w - \eta\,L'(w).$$

Пройдём один такой шаг до конца. Возьмём вес, который идёт от одного нейрона скрытого слоя к выходу. Пусть этот нейрон выдал $a=0{,}8$, на картинке восьмёрка ($y=1$), а сеть дала ей вероятность $p=0{,}3$. Для такого веса производная считается просто: разность между ответом сети и правильным ответом, умноженная на вход:

$$L'(w)=(p-y)\,a=(0{,}3-1)\cdot0{,}8=-0{,}56.$$

Производная отрицательна, значит ошибка уменьшится, если вес увеличить. Со скоростью обучения $\eta=0{,}1$ получаем

$$w \leftarrow w-0{,}1\cdot(-0{,}56)=w+0{,}056.$$

Проверим, что стало лучше. После такой поправки вероятность восьмёрки на этой картинке поднимается с $0{,}300$ до примерно $0{,}309$, а штраф падает с $1{,}204$ до $1{,}173$. Выигрыш крошечный — но шагов будет много, и правятся сразу все 1057 весов, каждый по своей производной.



*Рисунок 4 — Шаг против производной*

Размер шага приходится выбирать. Слишком маленький $\eta$ — и за отведённое число проходов сеть почти не успеет обучиться; слишком большой — и поправка перелетит через самое низкое место, а ошибка начнёт скакать. В `MLPClassifier` скорость обучения задаёт `learning_rate_init`, по умолчанию 0,001.



> **Примечание**
> 
> Для весов скрытого слоя производную считать сложнее: такой вес влияет на ответ не напрямую, а через следующий слой. Библиотека делает это сама, а способ называется **обратным распространением ошибки**; его разбор ждёт в главе о математике ИИ. Сейчас достаточно знать, что производная есть у каждого веса.


### Эпохи

Один полный проход по всем обучающим картинкам называют **эпохой**. Параметр `max_iter=200` задал 200 эпох. После каждой из них `MLPClassifier` запоминает среднюю ошибку в списке `loss_curve_`, и по этому списку видно, училась ли сеть.

In [ ]:
curve = net.loss_curve_  # ошибка после каждой эпохи
print(len(curve))
print([round(float(curve[i]), 3) for i in (0, 9, 49, 99, 199)])


Вывод:

```text
200
[0.742, 0.615, 0.33, 0.144, 0.052]
```

Ошибка уменьшилась с 0,742 до 0,052: сначала быстро, потом всё медленнее. Так и должно быть — пока веса случайны, поправлять есть что; ближе к концу сеть ошибается редко, и шаги становятся мелкими.



*Рисунок 5 — Ошибка по эпохам*

Убывающая кривая — признак того, что обучение идёт. Если ошибка стоит на месте, обучение не получилось: обычно виновата слишком маленькая скорость обучения. Если она прыгает — шаг слишком велик.

## А если цифр десять

Теперь вернём все цифры. С двумя классами достаточно одного выхода: вероятность второго класса равна $1-p$. С десятью классами сеть даёт **десять** выходов — по вероятности на каждую цифру, — а вероятности приводят к сумме 1. Такое преобразование называют **softmax**. Ответом считают цифру с наибольшей вероятностью.

В коде не меняется почти ничего: тот же `MLPClassifier`, тот же `fit`. Скрытый слой возьмём побольше — десять классов различать труднее, чем два.

In [ ]:
X_all = digits.data / 16
y_all = digits.target
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_all, y_all, test_size=0.3, random_state=0, stratify=y_all)

net10 = MLPClassifier(hidden_layer_sizes=(32,), max_iter=200, random_state=0)
net10.fit(Xa_train, ya_train)
print(round(net10.score(Xa_train, ya_train), 3))
print(round(net10.score(Xa_test, ya_test), 3))


Вывод:

```text
0.995
0.976
```

На 1257 обучающих картинках точность 99,5 %, на 540 отложенных — 97,6 %. Весов в этой сети $64\cdot32+32+32\cdot10+10=2410$.

Прочитаем ответ целиком — все десять вероятностей для первой проверочной картинки.

In [ ]:
proba10 = net10.predict_proba(Xa_test)
print(proba10[0].round(3))
print(round(float(proba10[0].sum()), 3))
print(net10.predict(Xa_test)[0], ya_test[0])


Вывод:

```text
[0.    0.91  0.    0.001 0.001 0.    0.    0.    0.087 0.   ]
1.0
1 1
```

Вероятности сложились в единицу, как и обещано. Наибольшая — 0,91 у цифры 1, её сеть и назвала; настоящий ответ тоже 1. Второй по величине ответ — 0,087 у восьмёрки: сеть немного колебалась между единицей и восьмёркой, и такие колебания видны по вероятностям.



*Рисунок 6 — Десять вероятностей для одной картинки*

Посмотрим, где сеть ошибается.

In [ ]:
pred10 = net10.predict(Xa_test)
wrong10 = np.where(pred10 != ya_test)[0]
print(len(wrong10))
for i in wrong10[:5]:
    print(ya_test[i], "→", pred10[i], round(float(proba10[i].max()), 2))


Вывод:

```text
13
8 → 1 0.79
1 → 8 0.67
2 → 1 0.89
9 → 5 0.78
9 → 3 0.42
```

Тринадцать ошибок на 540 картинок. В первом столбце настоящая цифра, во втором ответ сети, в третьем её уверенность. Ошибки не случайны: единица и восьмёрка путаются в обе стороны, девятка похожа то на пятёрку, то на тройку. На картинке 8 на 8 от цифры остаётся немного, и у некоторых написаний формы действительно совпадают.



*Рисунок 7 — Картинки, на которых сеть ошиблась*

## Честная проверка

Точность 0,976 сама по себе ничего не говорит. Сравним её с постоянным ответом и с логистической регрессией — простой моделью из первой главы — на тех же данных и том же разбиении.

In [ ]:
from sklearn.linear_model import LogisticRegression

constant10 = DummyClassifier(strategy="most_frequent").fit(Xa_train, ya_train)
logreg10 = LogisticRegression(max_iter=2000).fit(Xa_train, ya_train)
print(round(constant10.score(Xa_test, ya_test), 3))
print(round(logreg10.score(Xa_test, ya_test), 3))


Вывод:

```text
0.102
0.97
```

*Таблица 1 — Доля верных ответов на 540 отложенных картинках*

| Модель | Точность на тесте | Число ошибок |
| :--- | ---: | ---: |
| Постоянный ответ | 0,102 | 485 |
| Логистическая регрессия | 0,970 | 16 |
| Сеть 64–32–10 | 0,976 | 13 |

Сеть обошла линейную модель, но всего на три картинки. Это честный результат, и он полезнее красивого: цифры 8 на 8 — задача, с которой почти справляется и прямая граница, поэтому выигрывать здесь особо нечем. Сила сети проявляется там, где признаки сложнее, — на больших изображениях и текстах; этим мы займёмся в следующих главах. А на простой табличной задаче модели из первой главы остаются хорошим выбором.

Теперь посмотрим, что бывает, когда примеров мало. Обучим такую же сеть на первых 100 картинках вместо 1257.

In [ ]:
small = MLPClassifier(hidden_layer_sizes=(32,), max_iter=2000, random_state=0)
small.fit(Xa_train[:100], ya_train[:100])
print(round(small.score(Xa_train[:100], ya_train[:100]), 3))
print(round(small.score(Xa_test, ya_test), 3))


Вывод:

```text
1.0
0.893
```

На обучающих картинках сеть не ошиблась ни разу, на новых точность упала до 0,893. Разрыв между этими двумя числами называют **переобучением**: 2410 весов легко запоминают 100 примеров наизусть, но запомненное не переносится на новые. Поэтому точность на обучающей части — не результат, а только признак того, что обучение шло.



> **Важно**
> 
> Оценивайте сеть только на данных, которые не участвовали в обучении. Размер скрытого слоя и число эпох тоже нельзя подбирать по тестовой части: для этого отделяют ещё одну, проверочную часть, а тест оставляют на один раз в самом конце.


## Попробуйте сами

1. Обучите сеть на тройках и восьмёрках с `max_iter=50`, затем с `max_iter=500`. Сравните точность на тесте и объясните разницу. Для самопроверки: 0,963 и 0,991; при 500 обучение остановилось само на 386-й эпохе, потому что ошибка перестала уменьшаться.
2. Задайте той же сети `learning_rate_init=0.0001` при `max_iter=200` и посмотрите на `loss_curve_`. Для самопроверки: точность на тесте падает до 0,889, ошибка к концу остаётся около 0,52 — шаг слишком мал, 200 эпох не хватило.
3. Поставьте `hidden_layer_sizes=(1,)` на той же задаче. Почему результат близок к логистической регрессии? Для самопроверки: 0,963; сеть с одним нейроном в скрытом слое считает одну взвешенную сумму входов, поэтому её возможности почти как у линейной модели.
4. На десяти цифрах обучите сети со скрытым слоем из 4 и из 128 нейронов. Для самопроверки: 0,832 и 0,982 на тесте; четырём нейронам не хватает, чтобы различить десять цифр.
5. Вернитесь к задаче XOR из прошлого параграфа: обучите `MLPClassifier(hidden_layer_sizes=(4,), max_iter=5000, random_state=0)` на четырёх точках и проверьте предсказания. Повторите с `random_state=2`. Для самопроверки: при `random_state=0` все четыре ответа верны, при `random_state=2` — три из четырёх. Обучение начинается со случайных весов, и иногда спуск застревает; это ещё одна причина фиксировать `random_state` и проверять результат.

## Что стоит унести с собой

Картинку сеть получает как строку чисел: 8 на 8 точек превращаются в 64 признака, и числа на входе лучше делать небольшими. Обучение — это цикл: прогноз, ошибка, поправка всех весов против производной со шагом $\eta$; один проход по всем примерам называют эпохой, а падающая `loss_curve_` показывает, что цикл работает. Для двух классов у сети один выход с вероятностью, для десяти — десять выходов, сумма которых равна единице. Точность на обучающей части говорит лишь о том, что обучение шло: судить о модели можно только по отложенным данным и только в сравнении с отправной точкой и более простой моделью.

## Самопроверка




### 1. Сколько входов у сети, которая получает картинку 8 на 8 точек?

A. 8, по одному на строку картинки.

B. 16, по числу нейронов скрытого слоя.

C. 64, по одному на каждую точку.

D. 1797, по числу картинок в наборе.




### 2. Все 64 яркости и так измерены одинаково, от 0 до 16. Зачем их всё равно разделили на 16?

A. Чтобы выровнять признаки, измеренные в разных единицах.

B. Чтобы числа на входе были небольшими и сеть не начинала с очень уверенных неверных ответов.

C. Чтобы вероятности на выходе складывались в единицу.

D. Чтобы картинок стало меньше и обучение шло быстрее.




### 3. На картинке восьмёрка, а сеть дала ей вероятность 0,1. Чему равен штраф $-\ln p$ и как он изменится, если вероятность вырастет до 0,9?

A. Около 0,105; при 0,9 вырастет примерно до 2,303.

B. Около 2,303; при 0,9 останется таким же.

C. Около 2,303; при 0,9 упадёт примерно до 0,105.

D. Ровно 0,1; при 0,9 станет ровно 0,9.




### 4. Производная ошибки по весу $w$ отрицательна. Что сделает шаг обучения?

A. Увеличит $w$, потому что от веса отнимают производную со множителем $\eta$.

B. Уменьшит $w$, потому что производная показывает направление роста.

C. Оставит $w$ без изменений до следующей эпохи.

D. Обнулит $w$ и начнёт подбор заново.




### 5. Что такое эпоха обучения?

A. Один обработанный пример из обучающей части.

B. Один полный проход по всем обучающим примерам.

C. Одно изменение одного веса сети.

D. Одна проверка точности на отложенных данных.




### 6. Сеть различает десять цифр. Что верно про её десять выходов?

A. Каждый выход равен 0 или 1, и единица стоит ровно одна.

B. Выходы независимы, и каждый может быть равен 1 одновременно с другими.

C. Сумма выходов равна 10, по единице на класс.

D. Выходы — вероятности, их сумма равна 1, а ответ — наибольшая из них.




### 7. Зачем сравнивать точность сети с постоянным ответом, который дал 0,102?

A. Постоянный ответ может оказаться лучшей моделью для задачи.

B. Чтобы понять, сколько сеть заработала сама, а не получила из состава данных.

C. Чтобы выбрать размер скрытого слоя по результату на тесте.

D. Чтобы заменить проверку на отложенных данных.




### 8. На 100 обучающих картинках сеть дала точность 1,0, а на отложенных — 0,893. Как это называется и о чём говорит?

A. Хорошее обучение: на обучающей части ошибок нет.

B. Ошибка в разделении данных: такого разрыва быть не может.

C. Переобучение: 2410 весов запомнили 100 примеров, но запомненное не переносится на новые.

D. Слишком маленькая скорость обучения.




### 9. Ошибка в `loss_curve_` почти не уменьшается от эпохи к эпохе. Что проверить в первую очередь?

A. Размер тестовой части: он влияет на обучение.

B. Скорость обучения и масштаб признаков.

C. Число классов в задаче.

D. Порядок столбцов в обучающей таблице.




### 10. Зачем `MLPClassifier` задают `random_state`?

A. Он задаёт число эпох обучения.

B. Он перемешивает классы, чтобы они стали равными по размеру.

C. Обучение начинается со случайных весов, и `random_state` делает запуск повторяемым.

D. Он отключает случайность, и результат перестаёт зависеть от данных.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Картинку вытягивают в один вектор. Сколько в нём чисел?



**Решение**


**C.** Все строки картинки выписывают подряд, поэтому на входе $8\cdot8=64$ числа — по одному на точку. Число картинок в наборе и размер скрытого слоя к числу входов не относятся.



### Вопрос 2

**Подсказка**


Сравните начальную ошибку: 0,74 с делением и 2,46 без него.



**Решение**


**B.** Выравнивать признаки между собой не нужно — они с самого начала в одном масштабе. Деление уменьшает числа, которые попадают в сеть, поэтому обучение начинается с ошибки 0,74 вместо 2,46. На итоговую точность в этой задаче оно не повлияло; сумму вероятностей обеспечивает softmax, а число картинок не меняется.



### Вопрос 3

**Подсказка**


Штраф тем больше, чем меньше вероятность у правильного ответа.



**Решение**


**C.** $-\ln0{,}1\approx2{,}303$, а $-\ln0{,}9\approx0{,}105$. Уверенная ошибка обходится примерно в двадцать раз дороже уверенного верного ответа.



### Вопрос 4

**Подсказка**


Запишите $w \leftarrow w - \eta\,\partial L/\partial w$ и подставьте отрицательное число.



**Решение**


**A.** Вычитание отрицательной производной увеличивает вес. Отрицательная производная и означает, что при увеличении веса ошибка уменьшается.



### Вопрос 5

**Подсказка**


Чем задаётся длина списка `loss_curve_`?



**Решение**


**B.** Эпоха — проход по всем обучающим примерам; `max_iter=200` задаёт 200 таких проходов, и в `loss_curve_` оказывается 200 значений средней ошибки.



### Вопрос 6

**Подсказка**


Посмотрите на вывод `predict_proba` для одной картинки и сложите числа.



**Решение**


**D.** Softmax приводит десять чисел к сумме 1, поэтому их читают как вероятности классов; ответом считают цифру с наибольшей вероятностью.



### Вопрос 7

**Подсказка**


Что получилось бы, если бы в наборе 90 % картинок были одной цифрой?



**Решение**


**B.** Отправная точка показывает, какую точность даёт сам состав данных. На десяти примерно равных классах это около 0,1, поэтому 0,976 — результат модели; при перекошенных классах постоянный ответ может дать и 0,9.



### Вопрос 8

**Подсказка**


Сравните число весов сети с числом обучающих примеров.



**Решение**


**C.** Весов намного больше, чем примеров, поэтому сеть запоминает обучающую часть наизусть. Разрыв между точностью на обучении и на новых данных — обычный признак переобучения.



### Вопрос 9

**Подсказка**


Отчего шаг может оказаться слишком мелким для заметного изменения весов?



**Решение**


**B.** При слишком маленькой скорости обучения поправки почти не меняют веса. Слишком крупные входные числа мешают иначе: сеть начинает с большой ошибки и уверенных неверных ответов. Размер тестовой части в обучении не участвует, а порядок столбцов для сети безразличен, если он одинаков у всех примеров.



### Вопрос 10

**Подсказка**


Какими были 1057 весов до вызова `fit`?



**Решение**


**C.** Начальные веса случайны, поэтому два запуска без `random_state` дают разные результаты, а иногда спуск застревает, как в задании 5. Фиксированное значение делает запуск повторяемым, но не отменяет саму случайность начала.